# News In Perspective NLP Analysis

Shared notebook template for team analysis on exported app data.

Example notebook:
1. Export from Postgres with `pnpm export:kagi:notebook`
2. Sync the workspace to `MyDrive/NewsInPerspective`
3. Open `notebooks/nlp_analysis.ipynb` in Colab or run this template locally

Colab path requirement:
- This notebook expects the shared folder at `MyDrive/NewsInPerspective`
- If your Drive path differs, update `WORKSPACE_DIR` in the setup cell below
- This notebook expects `news.jsonl` next to the notebook file

In [6]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [7]:
# Mount Google Drive in Colab and point the notebook at the shared folder.
from google.colab import drive  # type: ignore

drive.mount("/content/drive")

WORKSPACE_DIR = Path("/content/drive/MyDrive/NewsInPerspective")

NEWS_FILE = WORKSPACE_DIR / "news.jsonl"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Load the single denormalized export file used throughout the notebook.
news_df = pd.read_json(NEWS_FILE, lines=True)
news_df["full_text"] = news_df["full_text"].fillna("")
news_df["analysis_text"] = news_df["analysis_text"].fillna("").str.strip()
news_df["published_at"] = pd.to_datetime(news_df["published_at"], errors="coerce")

# Backfill optional columns so older exports do not crash the example notebook.
for column in ["category", "region", "date"]:
    if column not in news_df.columns:
        news_df[column] = None

print(f"articles: {len(news_df)}")
print(f"articles with full text: {int(news_df['full_text_available'].sum())}")

/tmp/ipykernel_7190/2949796654.py:2: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  news_df = pd.read_json(NEWS_FILE, lines=True)


ValueError: Expected object or value

## Cluster View
Example: derive one cluster-level table from the article-level export.

In [ ]:
# Convert the exported date column into a real datetime before aggregation.
news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce")


def first_non_null(series: pd.Series):
    """Return the first non-null value in a series, if one exists."""
    non_null = series.dropna()
    return non_null.iloc[0] if not non_null.empty else None


# Build one row per cluster and derive the time window from the article dates.
clusters_df = (
    news_df.groupby("cluster_id", dropna=False)
    .agg(
        cluster_title=("cluster_title", "first"),
        cluster_source_count=("cluster_source_count", "max"),
        cluster_article_count=("cluster_article_count", "max"),
        category=("category", first_non_null),
        region=("region", first_non_null),
        date_from=("date", "min"),
        date_until=("date", "max"),
    )
    .reset_index()
)

print(f"clusters: {clusters_df['cluster_id'].nunique()}")

## Example: quick overview

In [ ]:
clusters_df.sort_values(["cluster_source_count", "cluster_article_count"], ascending=False).head(10)

In [ ]:
# Example: compare domain coverage and extracted-text volume.
news_df.groupby("domain").agg(
    article_count=("url", "count"),
    full_text_articles=("full_text_available", "sum"),
    mean_full_text_length=("full_text_length", "mean"),
).sort_values("article_count", ascending=False).head(15)

In [ ]:
# Example: inspect how much of the corpus has extracted full text.
news_df["full_text_available"].value_counts(dropna=False)

## Example: extraction quality

In [ ]:
# Example: group by extraction result to see where scraping worked or failed.
quality_df = (
    news_df.groupby("extraction_status")
    .agg(article_count=("url", "count"), mean_text_length=("full_text_length", "mean"))
    .sort_values("article_count", ascending=False)
)

display(quality_df)
quality_df["article_count"].plot(kind="bar", title="Extraction status distribution")
plt.ylabel("Articles")
plt.tight_layout()

## Example: failed extraction URLs
These are example source URLs where browser-based extraction failed, so teammates can inspect them manually.

In [ ]:
failed_examples_df = (
    news_df.loc[
        news_df["extraction_status"] == "FAILED",
        ["domain", "original_url", "final_url", "url", "extraction_error"],
    ]
    .drop_duplicates()
    .head(15)
    .reset_index(drop=True)
)

print(f"failed extraction examples: {len(failed_examples_df)}")
display(failed_examples_df)

## Example: bi-gram analysis

In [ ]:
# Build a text corpus from the denormalized analysis text column.
corpus = news_df["analysis_text"].loc[news_df["analysis_text"].str.len() > 0]

bigram_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(2, 2),
    min_df=2,
)
bigram_matrix = bigram_vectorizer.fit_transform(corpus)

bigram_counts = (
    pd.DataFrame(
        {
            "bigram": bigram_vectorizer.get_feature_names_out(),
            "count": bigram_matrix.sum(axis=0).A1,
        }
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

bigram_counts.head(25)

## Example: TF-IDF terms
This highlights terms that are distinctive in the current slice.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
tfidf_scores = tfidf_matrix.mean(axis=0).A1

tfidf_df = (
    pd.DataFrame(
        {
            "term": tfidf_vectorizer.get_feature_names_out(),
            "mean_tfidf": tfidf_scores,
        }
    )
    .sort_values("mean_tfidf", ascending=False)
    .reset_index(drop=True)
)

tfidf_df.head(30)

## Example: per-domain distinctive terms

In [ ]:
# Look at the most frequent domains in the export first.
top_domains = news_df["domain"].value_counts().head(5).index.tolist()
print("Top domains:", top_domains)

for domain in top_domains:
    # Restrict to one domain and drop empty rows before vectorization.
    domain_text = news_df.loc[news_df["domain"] == domain, "analysis_text"]
    domain_text = domain_text.loc[domain_text.str.len() > 0]
    if len(domain_text) == 0:
        print(f"\nSkipping {domain}: no analysis_text rows")
        continue

    try:
        domain_vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1,
            max_df=1.0,
        )
        domain_matrix = domain_vectorizer.fit_transform(domain_text)
        domain_scores = domain_matrix.mean(axis=0).A1
        domain_terms = (
            pd.DataFrame(
                {
                    "term": domain_vectorizer.get_feature_names_out(),
                    "score": domain_scores,
                }
            )
            .sort_values("score", ascending=False)
            .head(10)
        )
    except ValueError as error:
        print(f"\nSkipping {domain}: {error}")
        continue

    print(f"\nTop TF-IDF terms for {domain}")
    display(domain_terms)

## Suggested next analyses

- Compare bigrams or TF-IDF terms by category or region
- Measure domain-level framing differences on the same story
- Build your own sentiment model and compare it to the exported app sentiment
- Add topic modeling or embeddings once the dataset and workflow settle

## Entity Recognition

In [ ]:
import spacy
from collections import Counter

In [ ]:
# Load spaCy English NER model
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Use full_text where available; otherwise use analysis_text
news_df["ner_text"] = news_df["full_text"].fillna("")
news_df.loc[news_df["ner_text"].str.len() == 0, "ner_text"] = news_df["analysis_text"].fillna("")

In [ ]:
# Limit very long articles for speed
news_df["ner_text_short"] = news_df["ner_text"].str.slice(0, 5000)
print("Rows available for NER:", (news_df["ner_text_short"].str.len() > 0).sum())

In [ ]:
# Extract entities from articles
TARGET_LABELS = {"PERSON", "ORG", "GPE", "LOC", "DATE", "EVENT"}

def extract_entities(text):
    """
    Extract named entities from article text using spaCy.
    Returns a list of dictionaries.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []

    doc = nlp(text)

    entities = []
    for ent in doc.ents:
        if ent.label_ in TARGET_LABELS:
            entities.append({
                "entity_text": ent.text.strip(),
                "entity_label": ent.label_
            })

    return entities

In [ ]:
# Apply NER
news_df["entities"] = news_df["ner_text_short"].apply(extract_entities)
news_df[["cluster_id", "cluster_title", "domain", "entities"]].head()

In [ ]:
# Convert article-level entities into a table
entity_rows = []

for idx, row in news_df.iterrows():
    for ent in row["entities"]:
        entity_rows.append({
            "article_index": idx,
            "cluster_id": row.get("cluster_id"),
            "cluster_title": row.get("cluster_title"),
            "domain": row.get("domain"),
            "region": row.get("region"),
            "url": row.get("url"),
            "entity_text": ent["entity_text"],
            "entity_label": ent["entity_label"]
        })

entities_df = pd.DataFrame(entity_rows)

print("Extracted entities:", len(entities_df))
entities_df.head(20)

Overall entity distribution

In [ ]:
entity_label_counts = (
    entities_df["entity_label"]
    .value_counts()
    .reset_index()
)

entity_label_counts.columns = ["entity_label", "count"]

display(entity_label_counts)

In [ ]:
entity_label_counts.plot(
    kind="bar",
    x="entity_label",
    y="count",
    legend=False,
    title="Named Entity Type Distribution"
)

plt.xlabel("Entity type")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Top entities overall
top_entities_overall = (
    entities_df
    .groupby(["entity_text", "entity_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

display(top_entities_overall)

In [ ]:
# Cluster-level entity summary
cluster_entities = (
    entities_df
    .groupby(["cluster_id", "cluster_title", "entity_text", "entity_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["cluster_id", "count"], ascending=[True, False])
)

cluster_entities.head(30)

In [ ]:
# Show top entities for the largest cluster
largest_cluster_id = (
    news_df.groupby("cluster_id")
    .size()
    .sort_values(ascending=False)
    .index[0]
)

largest_cluster_entities = (
    cluster_entities[cluster_entities["cluster_id"] == largest_cluster_id]
    .head(20)
)

print("Largest cluster:", largest_cluster_id)
display(largest_cluster_entities)

In [ ]:
# Domain-level entity comparison
domain_entities = (
    entities_df
    .groupby(["domain", "entity_text", "entity_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["domain", "count"], ascending=[True, False])
)

domain_entities.head(30)

In [ ]:
# Compare top entities in the top 5 domains
top_domains = news_df["domain"].value_counts().head(5).index.tolist()

for domain in top_domains:
    print(f"\nTop entities for domain: {domain}")
    display(
        domain_entities[domain_entities["domain"] == domain]
        .head(10)
    )

In [ ]:
# Cluster + domain comparison
cluster_domain_entities = (
    entities_df
    .groupby(["cluster_id", "cluster_title", "domain", "entity_text", "entity_label"])
    .size()
    .reset_index(name="count")
    .sort_values(["cluster_id", "domain", "count"], ascending=[True, True, False])
)

cluster_domain_entities.head(30)

In [ ]:
# Inspect one selected cluster across domains
selected_cluster_id = largest_cluster_id

selected_cluster_domain_entities = (
    cluster_domain_entities[
        cluster_domain_entities["cluster_id"] == selected_cluster_id
    ]
)

for domain in selected_cluster_domain_entities["domain"].dropna().unique()[:5]:
    print(f"\nCluster {selected_cluster_id} - Domain: {domain}")
    display(
        selected_cluster_domain_entities[
            selected_cluster_domain_entities["domain"] == domain
        ].head(10)
    )

In [ ]:
# Export NER results for app
OUTPUT_DIR = WORKSPACE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

entities_df.to_csv(OUTPUT_DIR / "article_level_entities.csv", index=False)
cluster_entities.to_csv(OUTPUT_DIR / "cluster_level_entities.csv", index=False)
domain_entities.to_csv(OUTPUT_DIR / "domain_level_entities.csv", index=False)
cluster_domain_entities.to_csv(OUTPUT_DIR / "cluster_domain_entities.csv", index=False)

print("NER outputs saved to:", OUTPUT_DIR)

JSON-ready cluster entity output

In [ ]:
def top_entities_for_cluster(group, n=10):
    return (
        group.sort_values("count", ascending=False)
        .head(n)[["entity_text", "entity_label", "count"]]
        .to_dict("records")
    )

cluster_entity_json_df = (
    cluster_entities
    .groupby(["cluster_id", "cluster_title"])
    .apply(top_entities_for_cluster)
    .reset_index(name="top_entities")
)

cluster_entity_json_df.head()

In [ ]:
cluster_entity_json = cluster_entity_json_df.to_dict("records")

with open(OUTPUT_DIR / "cluster_entities.json", "w", encoding="utf-8") as f:
    json.dump(cluster_entity_json, f, ensure_ascii=False, indent=2)

print("JSON file saved:", OUTPUT_DIR / "cluster_entities.json")